In [4]:
import io
import numpy as np
import note_seq
import pickle
import gin

DEFAULT_SAMPLE_RATE = 16000

def audio_bytes_to_np(path,
                      sample_rate=DEFAULT_SAMPLE_RATE,
                      normalize_db=0.1,
                      mono=True):
    with io.open(path, 'rb') as f:
        wave_data = f.read()
    return note_seq.audio_io.wav_data_to_samples_pydub(
      wav_data=wave_data, sample_rate=sample_rate, normalize_db=normalize_db,
      num_channels=1 if mono else None)

In [ ]:
import numpy as np
import io
import base64
from scipy.io import wavfile
from IPython.display import HTML, display

def play_audio(audio, sample_rate=16000, autoplay=False, filename='audio.wav'):
    """Play a 1D or 2D float32 audio array inline in Jupyter Notebook."""
    if len(audio.shape) == 2:
        audio = audio[0]

    # Clip and convert to int16
    audio = np.clip(audio, -1.0, 1.0)
    audio_int16 = (audio * 32767).astype(np.int16)

    buffer = io.BytesIO()
    wavfile.write(buffer, sample_rate, audio_int16)
    encoded = base64.b64encode(buffer.getvalue()).decode('ascii')

    html = f"""
    <audio controls {'autoplay' if autoplay else ''}>
        <source src="data:audio/wav;base64,{encoded}" type="audio/wav">
        Your browser does not support the audio element.
    </audio>
    <a download="{filename}" href="data:audio/wav;base64,{encoded}">⬇️ Download {filename}</a>
    """
    display(HTML(html))

In [ ]:
import ddsp
import ddsp.training
from ddsp.training.preprocessing import F0LoudnessPreprocessor
import tensorflow as tf
import librosa

# Increase Freq
# Mono - f0?

audio_path = '/home/jianzhis/demucs_out/jamendo/mdx_extra/1208527/vocals.wav'
audio = audio_bytes_to_np(audio_path)
if len(audio.shape) == 1:
    audio = audio[np.newaxis, :]
sr = 16000
cut = sr * 20
max_samples = sr * 5
audio = audio[:, cut:cut + max_samples]
play_audio(audio, sample_rate=sr, filename='orig.wav')
wav = audio[0]
wav_shifted = librosa.effects.pitch_shift(wav, sr=sr, n_steps=8)
audio = np.expand_dims(wav_shifted, 0)

In [7]:
audio_features = ddsp.training.metrics.compute_audio_features(audio)
audio_features['loudness_db'] = tf.cast(audio_features['loudness_db'], tf.float32)
audio_features_mod = None
play_audio(audio, sample_rate=sr, filename='shifted.wav')

2025-07-09 14:11:13.862625: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-07-09 14:11:13.862897: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-07-09 14:11:13.863126: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2025-07-09 14:11:13.863366: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-07-09 14:11:13.863433: W tensorflow/compiler/xla/stre

In [8]:
base_dir = '/home/jianzhis/ddsp/trumpet/'
checkpoint_path = base_dir
gin_file = base_dir + 'operative_config-0.gin'
dataset_file = base_dir + 'dataset_statistics.pkl'
with tf.io.gfile.GFile(dataset_file, 'rb') as f:
    dataset_statistics = pickle.load(f)
with gin.unlock_config():
  gin.parse_config_file(gin_file, skip_unknown=True)

time_steps_train = gin.query_parameter('F0LoudnessPreprocessor.time_steps')
n_samples_train = gin.query_parameter('Harmonic.n_samples')
hop_size = int(n_samples_train / time_steps_train)

time_steps = int(audio.shape[1] / hop_size)
n_samples = time_steps * hop_size

for key in ['f0_hz', 'f0_confidence', 'loudness_db']:
  audio_features[key] = audio_features[key][:time_steps]
audio_features['audio'] = audio_features['audio'][:, :n_samples]

model = ddsp.training.models.Autoencoder()
model.restore(checkpoint_path)


In [9]:
outputs = model(audio_features, training=False)
audio_gen = model.get_audio_from_outputs(outputs)


/home/jianzhis/.conda/envs/demucs/lib/python3.9/site-packages/librosa/core/convert.py:1869: RuntimeWarning: divide by zero encountered in log10
  + 2 * np.log10(f_sq)


In [10]:
play_audio(audio_gen.numpy()[0], sample_rate=DEFAULT_SAMPLE_RATE, filename='gen.wav')